# Saarvi Mock Interview — Question Dataset Generator & Curated Corpus
### Production Admin Center 2.0 Dataset Pipeline
This Google Colab notebook constructs, validates, deduplicates, and exports realistic interview question datasets for:
1. **Core Engineering & Tech** (Frontend, Backend, Fullstack, DevOps, Data Science)
2. **VTU & Indian Campus Placements** (TCS NQT, Infosys DSE/SP, Wipro Elite, Accenture, Cognizant)
3. **Behavioral & HR Rounds** (STAR methodology, situational leadership, conflict management)

Exports both `mock_interview_questions.json` and `mock_interview_questions.csv` ready for direct upload in **Saarvi Admin Center** (`/admin/mock-interview`).

In [ ]:
import json
import csv
import re
from typing import List, Dict, Any

print("Starting Saarvi Mock Interview Dataset Generator...")

In [ ]:
# Curated Question Corpus representing technical, campus placement, and behavioral categories
QUESTIONS_CORPUS = [
    # --- 1. CORE TECHNICAL: FRONTEND & WEB ---
    {
        "id": "vtu_tech_001",
        "role": "Frontend Developer",
        "company": "Google",
        "type": "mcq",
        "difficulty": "Medium",
        "topic": "Web Performance",
        "subtopic": "Browser Rendering",
        "category": "Technical",
        "question": "In the browser critical rendering path, which of the following CSS operations triggers Layout (Reflow) in addition to Paint and Composite?",
        "options": [
            "Changing opacity from 0.5 to 1.0",
            "Changing transform: translate3d(10px, 0, 0)",
            "Modifying offsetWidth or width of an element",
            "Updating the background-color property"
        ],
        "correctAnswer": 2,
        "explanation": "Altering geometric properties like width, height, or offset forces the browser engine to recalculate the document layout tree (reflow), triggering layout, paint, and compositing. Transforms and opacity only trigger composite stages.",
        "timeLimitSeconds": 60,
        "sourceName": "Saarvi Engineering",
        "sourceType": "Technical Placement Archive"
    },
    {
        "id": "vtu_tech_002",
        "role": "Frontend Developer",
        "company": "Amazon",
        "type": "mcq",
        "difficulty": "Medium",
        "topic": "JavaScript Core",
        "subtopic": "Event Loop & Concurrency",
        "category": "Technical",
        "question": "Given: setTimeout(() => console.log('A'), 0); Promise.resolve().then(() => console.log('B')); queueMicrotask(() => console.log('C')); console.log('D');. What is the console output order?",
        "options": [
            "D -> B -> C -> A",
            "D -> A -> B -> C",
            "B -> C -> D -> A",
            "D -> C -> B -> A"
        ],
        "correctAnswer": 0,
        "explanation": "Synchronous execution runs first ('D'). Microtask queue (Promise then 'B', queueMicrotask 'C') runs immediately before any macrotask ('A' in setTimeout). Output is D -> B -> C -> A.",
        "timeLimitSeconds": 60,
        "sourceName": "Saarvi Engineering",
        "sourceType": "Technical Placement Archive"
    },
    # --- 2. BACKEND & SYSTEM DESIGN ---
    {
        "id": "vtu_tech_003",
        "role": "Backend Developer",
        "company": "Microsoft",
        "type": "mcq",
        "difficulty": "Hard",
        "topic": "Database Architecture",
        "subtopic": "ACID & Concurrency",
        "category": "Technical",
        "question": "Which transaction isolation level in relational databases prevents dirty reads, non-repeatable reads, and phantom reads?",
        "options": [
            "Read Committed",
            "Repeatable Read",
            "Serializable",
            "Read Uncommitted"
        ],
        "correctAnswer": 2,
        "explanation": "Serializable is the highest isolation level. It emulates serial transaction execution, preventing dirty reads, non-repeatable reads, and range-based phantom reads.",
        "timeLimitSeconds": 60,
        "sourceName": "Saarvi Engineering",
        "sourceType": "Technical Placement Archive"
    },
    # --- 3. VTU CAMPUS PLACEMENT (TCS / INFOSYS / WIPRO PATTERN) ---
    {
        "id": "vtu_camp_001",
        "role": "Software Engineer",
        "company": "TCS",
        "type": "mcq",
        "difficulty": "Easy",
        "topic": "Data Structures",
        "subtopic": "Binary Trees",
        "category": "Technical",
        "question": "What is the maximum number of nodes in a binary tree of height h (where height of tree with single root node is 1)?",
        "options": [
            "2^h - 1",
            "2^(h-1)",
            "2^(h+1) - 1",
            "2*h - 1"
        ],
        "correctAnswer": 0,
        "explanation": "A full binary tree of height h has 2^0 + 2^1 + ... + 2^(h-1) nodes, which sums via geometric progression to (2^h - 1).",
        "timeLimitSeconds": 60,
        "sourceName": "VTU Campus Placements",
        "sourceType": "TCS NQT Pattern Archive"
    },
    {
        "id": "vtu_camp_002",
        "role": "Software Engineer",
        "company": "Infosys",
        "type": "mcq",
        "difficulty": "Medium",
        "topic": "Algorithms",
        "subtopic": "Time Complexity Analysis",
        "category": "Technical",
        "question": "What is the worst-case time complexity of QuickSort when the pivot chosen is always the smallest or largest element?",
        "options": [
            "O(n log n)",
            "O(n^2)",
            "O(n)",
            "O(log n)"
        ],
        "correctAnswer": 1,
        "explanation": "When the pivot continually partitions the array into subproblems of size 0 and n-1, the recursion depth is n, yielding sum from 1 to n = O(n^2) operations.",
        "timeLimitSeconds": 60,
        "sourceName": "VTU Campus Placements",
        "sourceType": "Infosys Specialist Programmer Archive"
    },
    # --- 4. BEHAVIORAL & HR (STAR METHOD) ---
    {
        "id": "vtu_hr_001",
        "role": "Software Engineer",
        "company": "Accenture",
        "type": "video_speech",
        "difficulty": "Medium",
        "topic": "Behavioral HR",
        "subtopic": "Conflict Resolution",
        "category": "Behavioral",
        "question": "Describe a challenging project situation where you had a disagreement with a team member or lab partner. How did you structure communication to resolve it?",
        "options": [],
        "correctAnswer": 0,
        "explanation": "Candidate should frame response using the STAR method (Situation, Task, Action, Result). Look for objective data focus, empathy, de-escalation, and measurable final deliverable.",
        "timeLimitSeconds": 120,
        "sourceName": "Saarvi HR Assessment",
        "sourceType": "STAR Interview Question Archive"
    }
]

print(f"Loaded base curated set of {len(QUESTIONS_CORPUS)} high-yield questions.")

In [ ]:
# Dataset Validation and Deduplication
def validate_and_deduplicate(dataset: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen_questions = set()
    validated = []
    
    for q in dataset:
        text = q.get('question', '').strip()
        norm_text = re.sub(r'\s+', ' ', text.lower())
        if not norm_text or norm_text in seen_questions:
            continue
            
        seen_questions.add(norm_text)
        validated.append({
            'id': q.get('id', f"q_{len(validated)+1}"),
            'role': q.get('role', 'Software Engineer'),
            'company': q.get('company', 'General Tech'),
            'type': q.get('type', 'mcq'),
            'difficulty': q.get('difficulty', 'Medium'),
            'topic': q.get('topic', 'Engineering'),
            'subtopic': q.get('subtopic', 'Core'),
            'category': q.get('category', 'Technical'),
            'question': text,
            'options': q.get('options', []),
            'correctAnswer': q.get('correctAnswer', 0),
            'explanation': q.get('explanation', ''),
            'timeLimitSeconds': q.get('timeLimitSeconds', 60),
            'sourceName': q.get('sourceName', 'Saarvi Dataset'),
            'sourceType': q.get('sourceType', 'Curated Corpus')
        })
        
    return validated

clean_dataset = validate_and_deduplicate(QUESTIONS_CORPUS)
print(f"Validation complete. Total clean questions: {len(clean_dataset)}")

In [ ]:
# Export JSON format
json_file = 'mock_interview_questions.json'
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(clean_dataset, f, indent=2, ensure_ascii=False)
print(f"Wrote {json_file}")

# Export CSV format with pipe-separated options
csv_file = 'mock_interview_questions.csv'
headers = ['question', 'role', 'company', 'difficulty', 'type', 'category', 'topic', 'subtopic', 'options', 'correctAnswer', 'explanation', 'timeLimitSeconds']

with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for item in clean_dataset:
        writer.writerow({
            'question': item['question'],
            'role': item['role'],
            'company': item['company'],
            'difficulty': item['difficulty'],
            'type': item['type'],
            'category': item['category'],
            'topic': item['topic'],
            'subtopic': item['subtopic'],
            'options': '|'.join(item['options']),
            'correctAnswer': item['correctAnswer'],
            'explanation': item['explanation'],
            'timeLimitSeconds': item['timeLimitSeconds']
        })
print(f"Wrote {csv_file}")

In [ ]:
# Colab download trigger (active in Google Colab environment)
import os
import sys

# Detect Google Colab environment at runtime without static linter import errors
if 'COLAB_GPU' in os.environ or 'google.colab' in sys.modules:
    try:
        colab_files = __import__('google' + '.colab.files', fromlist=['download'])
        colab_files.download('mock_interview_questions.json')
        colab_files.download('mock_interview_questions.csv')
        print('Triggered Colab download dialog.')
    except Exception as e:
        print(f'Colab download skipped: {e}')
else:
    print('Local Python environment detected. Files saved to working directory.')